In [1]:
#| default_exp core

In [ ]:
#| export
import torch
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True
import regex as re

In [2]:
#| export
import os
from os import environ

def init_instance():
    os.environ["USE_DEEPSPEED"] = "1"
    os.environ["MASTER_PORT"]=str(6000+int(environ.get('INSTANCE', '0')))
    model_path = environ.get('MODEL', 'poetry')
    flavor_id = model_path + environ.get('CUDA_VISIBLE_DEVICES', '0') + environ.get('INSTANCE', '0')
    from tendo import singleton
    me = singleton.SingleInstance(flavor_id=flavor_id)
    return me, model_path

In [ ]:
#| export
from rest.storage import logs, connection
from sqlalchemy.dialects.postgresql import insert

def log(request, type, hash, log):
    sessionid = request.cookies.get('sessionid')
    insert_stmt = insert(logs).values(ip=request.headers.get('X-Real-IP'), origin=request.headers.get('Origin'),
                                      agent=request.headers.get('User-Agent'), fs=request.headers.get('X-Forwarded-Server'),
                                      ff=request.headers.get('X-Forwarded-For'),
                                      session=f'{sessionid}', type=type, hash=hash, log=str(log))
    connection.execute(insert_stmt)

In [ ]:
#| export
import regex as re

def fix_string(string) -> str:
    in_word = string
    in_between_words = ['-', '–']
    in_sentences = ['«', '(', '[', '{', '"', '„', '\'']

    for item in in_between_words:
        regex = r'\w[%s]\s\w' % item
        in_word = re.findall(regex, string)

        for x in in_word:
            a = x[:1]; b = x[3:4]
            string = string.replace(x, a + '-' + b)

    for item in in_sentences:
        string = string.replace(f' {item} ', f' {item}')

    return string

def process_seq(generated_sequences):
    reg_text = [re.match(r'[\w\W]*[\.!?]\n', item) for item in generated_sequences]
    reg_text2 = [re.match(r'[\w\W]*[\.!?]', item) for item in generated_sequences]
    result = [reg_item[0] if reg_item else reg_item2[0] if reg_item2 else item for reg_item, reg_item2, item in zip(reg_text, reg_text2, generated_sequences)]
    result = [fix_string(s) for s in result]
    return result 